In [4]:
!pip install --upgrade ipywidgets
!jupyter nbextension enable --py widgetsnbextension

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 31.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.2 MB/s  0:00:00
ERROR: Could not install packages due to an OSError: [Errno 30] Read-only file system: '/opt/conda/lib/python3.10/site-packages/widgetsnbextension'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [widgetsnbextension]
/bin/bash: /home/chengjie.zheng001/.local/bin/jupyter: /usr/bin/python3: bad interpreter: No such file or directory


In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # 如果 notebooks 在项目根目录下的 notebooks/ 下

In [5]:
import torch
from torch.utils.data import DataLoader

from time_moe.datasets.mirage_cmapss_dataset import (
    MirageCmapssWindowDataset, EntityBatchSampler, mirage_collate_fn
)
from time_moe.models.modeling_mirage import TimeMoeForMirage
from time_moe.models.modeling_time_moe import TimeMoeConfig

# dataset / loader
L = 64
stride = 16
ds = MirageCmapssWindowDataset(
    hf_name="nominal-io/nasa-turbofan-degradation",
    split="train",
    context_length=L,
    stride=stride,
    normalize=True,
)

B, K = 8, 4
sampler = EntityBatchSampler(ds.entity_to_indices, batch_entities=B, windows_per_entity=K, num_batches=10, seed=9899)
loader = DataLoader(ds, batch_sampler=sampler, collate_fn=mirage_collate_fn, num_workers=0)
batch = next(iter(loader))

device = "cuda" if torch.cuda.is_available() else "cpu"
batch = {k: v.to(device) for k, v in batch.items()}

# model config (tiny sanity model)
cfg = TimeMoeConfig(
    input_size=24,
    hidden_size=128,
    intermediate_size=512,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=4,
    num_experts=4,
    num_experts_per_tok=2,
    max_position_embeddings=512,

    # MoE backbone（必须，否则没有 routing）
    use_dense=False,

    # 必须 True，满足 config 的 XOR 断言
    apply_aux_loss=True,

    # 如果现在不想加 MoE balancing 的 loss，就把系数置 0
    router_aux_loss_factor=0.0,

    horizon_lengths=[1],
    _attn_implementation="eager",

    # stage-conditioned routing ON
    enable_stage_routing=True,
    stage_routing_num_fourier=8,
    stage_routing_max_freq=10.0,

    # MIRAGE heads
    mirage_proj_dim=128,
    mirage_pooling="last",
    mirage_rank_beta=0.02,
    mirage_learn_beta=False,
    mirage_use_pair_weight=True,
    mirage_num_prototypes=10,
    mirage_tau=0.1,
    mirage_proto_momentum=0.95,
    mirage_update_prototypes=True,
    mirage_lambda_rank=0.1,
    mirage_lambda_align=0.05,
)


m = TimeMoeForMirage(cfg).to(device)
m.train()

out = m(**batch)

print("loss:", float(out.loss))
print("loss_gen:", float(out.loss_gen), "loss_rank:", float(out.loss_rank), "loss_align:", float(out.loss_align))
print("z mean/std:", float(out.z.mean()), float(out.z.std()))
print("u norm mean:", float(out.u.norm(dim=-1).mean()))
print("prototype norm mean:", float(m.prototypes.norm(dim=-1).mean()))
print("bin_ids unique:", out.bin_ids.unique().numel() if out.bin_ids is not None else None)


loss: 0.41452836990356445
loss_gen: 0.2547208070755005 loss_rank: 0.2977921664714813 loss_align: 2.6005663871765137
z mean/std: -0.017506446689367294 0.024777483195066452
u norm mean: 1.0
prototype norm mean: 1.0
bin_ids unique: 10
